In [7]:
import os
import re
import string
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter  
from sentence_transformers import SentenceTransformer
import chromadb
from groq import Groq

load_dotenv()
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "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")


# PART 1: Why Chat History Matters


In [8]:
print("=" * 55)
print("PART 1: The Problem Without Chat History")
print("=" * 55)

# Without chat history — every question is independent
# User: "What is supervised learning?"
# AI:   "Supervised learning is..."
# User: "What are its applications?"  ← WHO IS "ITS"???
# AI:   ??? model has no idea what "its" refers to

# With chat history — context carries over
# User: "What is supervised learning?"
# AI:   "Supervised learning is..."
# User: "What are its applications?"
# AI:   knows "its" = supervised learning → answers correctly

# 🔑 This is called "conversational context" and it's what
# separates a chatbot from a proper AI assistant

PART 1: The Problem Without Chat History


# PART 2: Setup — Load PDF + ChromaDB (same as Day 2)

In [32]:
print("\n" + "=" * 55)
print("PART 2: Setup")
print("=" * 55)

PDF_PATH="document.pdf"

#Load and chunk PDF
loader=PyPDFLoader(PDF_PATH)
pages=loader.load()#converts each page into langchain document like [
#    Document(page_content="Text from page 1...", metadata={...}),
#    Document(page_content="Text from page 2...", metadata={...}),
#    Document(page_content="Text from page 3...", metadata={...}),
#]

splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,chunk_overlap=50
)
chunks=splitter.split_documents(pages)#split to chucnks as content is too large for embeddings ,search etc
#[
  # Document(page_content="chars 0-500", metadata={"page":0}),
    #Document(page_content="chars 450-950", metadata={"page":0}),
  # Document(page_content="chars 900-1400", metadata={"page":0}),
#]
print(f"✅ Loaded {len(pages)} pages, {len(chunks)} chunks")


#Embeddings(Convert the chunks to embeddings)
print("Loading embedding model...")
embedder=SentenceTransformer("all-MiniLM-L6-v2")
texts=[c.page_content for c in chunks]
embeddings=embedder.encode(texts);
print(embeddings)
print("✅ Embeddings ready!")


#chromadb
client =chromadb.PersistentClient(path="./rag_day3_db")
try:
    client.delete_collection("day3_docs")
except:
    pass
collection=client.create_collection("day3_docs")
collection.add(
    documents=texts,#chunks->content
    embeddings=embeddings.tolist(),
    metadatas=[
        {"page":c.metadata.get("page",0)+1} for c in chunks
    ],
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)
print(f"✅ ChromaDB ready with {collection.count()} chunks!")
groq_client = Groq(api_key="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")






PART 2: Setup
✅ Loaded 16 pages, 25 chunks
Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5287.35it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[-0.08817074 -0.00769602 -0.02041199 ...  0.05969208 -0.06583954
   0.02676666]
 [ 0.07401869 -0.02656193 -0.04802348 ... -0.03195199  0.02682048
  -0.00631388]
 [-0.03423412 -0.01896102 -0.03554009 ...  0.00521962  0.03364883
   0.00202118]
 ...
 [ 0.00196202 -0.03880864  0.03117282 ...  0.124368    0.00524877
  -0.01095609]
 [-0.06644998  0.02640102  0.02184393 ...  0.12268586 -0.00414273
  -0.00292628]
 [-0.08178771  0.00341649  0.02532803 ...  0.05614989  0.06970598
  -0.03263439]]
✅ Embeddings ready!
✅ ChromaDB ready with 25 chunks!


# PART 3: Understanding Chat History Format


In [33]:
print("\n" + "=" * 55)
print("PART 3: Chat History Format")
print("=" * 55)
# Chat history is just a list of messages
# Each message has a role and content
# role = "user"      → what the user said
# role = "assistant" → what the AI said

# Example history:
example_history = [
    {
        "role":    "user",
        "content": "What is supervised learning?"
    },
    {
        "role":    "assistant",
        "content": "Supervised learning is a type of ML where the model learns from labeled data..."
    },
    {
        "role":    "user",
        "content": "What are its applications?"  # "its" = supervised learning
    }
]

# When we send this full history to the LLM, it understands
# that "its" refers to supervised learning from message 1
# This is exactly how ChatGPT works internally

print("Chat history format:")
for msg in example_history:
    print(f"  {msg['role'].upper()}: {msg['content'][:60]}...")







PART 3: Chat History Format
Chat history format:
  USER: What is supervised learning?...
  ASSISTANT: Supervised learning is a type of ML where the model learns f...
  USER: What are its applications?...


# PART 4: RAG with Chat History — Step by Step


In [37]:
print("\n" + "=" * 55)
print("PART 4: RAG + Chat History")
print("=" * 55)

def retrieve_chunks(question,n_results=3):
    q_embedding=embedder.encode([question]).tolist()
    results=collection.query(
        query_embeddings=q_embedding,
        n_results=n_results,
        include=["documents","metadatas","distances"]
    )
    chunks_found=[]
    for i in range(len(results["documents"][0])):
        chunks_found.append({
            "text":results["documents"][0][i],
            "page":results["metadatas"][0][i]["page"],
            "relevence":round(1-results["distances"][0][i],2)
        })

    return chunks_found

def ask_with_history(question,chat_history,n_chunks=3):
    """
    RAG pipeline with conversation memory.

    How it works:
    1. Retrieve relevant chunks for current question
    2. Build context from chunks
    3. Build messages list = system prompt + history + new question
    4. Send to LLM — it sees full conversation
    5. Get answer + add to history
    6. Return answer + updated history
    """
    chunks_found=retrieve_chunks(question,n_chunks)
    context=""#Because LLMs only understand plain text prompts.
    sources=[]
    for chunk in chunks_found:
        context+=f"[Page {chunk['page']}]:{chunk['text']}\n\n"
        sources.append(f"Page{chunk['page']}")
     
    # Step 3: Build messages
    # 🔑 Structure:
    # - system message: tells LLM who it is + gives context
    # - chat_history:   all previous Q&A pairs
    # - new question:   current user question
    system_message = {
        "role": "system",
        "content": f"""You are a helpful assistant that answers questions about documents.
Use ONLY the provided context to answer. Always cite page numbers.
If the context doesn't contain enough information, say so clearly.
Keep answers concise and clear.

Relevant context from document:
{context}"""
    }

    #build full msglist
    messages=[system_message] + chat_history + [
        {"role":"user","content":question}
    ]
    response = groq_client.chat.completions.create(
        model       = "llama-3.3-70b-versatile",
        messages    = messages,
        max_tokens  = 400,
        temperature = 0.1
    )
    answer=response.choices[0].message.content
    
    # Step 5: Update chat history
    # Add current Q&A to history for next question
    updated_history=chat_history + [
        {"role":"user", "content":question},
        {"role":"assistant" , "content":answer}
    ]

    return {
        "answer":  answer,
        "sources": sources,
        "history": updated_history
    }




PART 4: RAG + Chat History


# PART 5: Test Conversation — See History in Action


In [44]:
print("\n" + "=" * 55)
print("PART 5: Test Conversation")
print("=" * 55)

history=[]
print("\n" + "─" * 50)
print("Q1: What is package diagram?")
result1  = ask_with_history("What is package diagram?", history)
history  = result1["history"]  # update history
print(f"A: {result1['answer']}")
print(f"Sources: {result1['sources']}")


print("\n" + "─" * 50)
print("Q2: What is use of it in stock trading?")  # "it" = machine learning
result2  = ask_with_history(" What is use of it in stock trading?", history)
history  = result2["history"]
print(f"A: {result2['answer']}")
print(f"Sources: {result2['sources']}")


PART 5: Test Conversation

──────────────────────────────────────────────────
Q1: What is package diagram?
A: According to page 2, the Package Diagram represents the "high-level organization of the system by grouping related components into logical packages." It illustrates how different layers of the Stock Trading & Risk Management System interact with each other.
Sources: ['Page2', 'Page4', 'Page2']

──────────────────────────────────────────────────
Q2: What is use of it in stock trading?
A: The package diagram provides a structured view of the stock trading system using layered architecture (Page 4). It ensures scalability, security, and modular design, making it suitable for real-world financial systems.
Sources: ['Page3', 'Page6', 'Page4']


# PART 6: History Management
# Important — history grows forever without limitsy

In [52]:
print("\n" + "=" * 55)
print("PART 6: History Management")
print("=" * 55)
# Problem: if user asks 100 questions, history gets huge
# LLMs have a context window limit — too much history = error
# Solution: keep only last N messages

def trim_history(history,max_messages=10):
    """
    Keep only the last max_messages in history.
    Always keep pairs (user + assistant) so we don't
    cut in the middle of a conversation turn.
    max_messages=10 means 5 Q&A pairs
    """
    if len(history) <= max_messages:
        return history
    
    # Keep last max_messages (always even number for pairs)
    trimmed=history[-max_messages:]

    # Make sure we start with a user message not assistant
    if trimmed[0]["role"] == "assistant":
        trimmed = trimmed[1:]

    return trimmed

long_history=history *5;
print(f"Before trim: {len(long_history)} messages")
trimmed = trim_history(long_history, max_messages=6)
print(f"After trim:  {len(trimmed)} messages")
print("✅ History trimming works!")



PART 6: History Management
Before trim: 20 messages
After trim:  6 messages
✅ History trimming works!


# PART 7: Clean RAGChat Class
# This is what your Flask API will use

In [54]:
print("\n" + "=" * 55)
print("PART 7: Clean RAGChat Class")
print("=" * 55)

class RAGChat:
    def __init__(self,pdf_path,max_history=10):
        print(f"Initializing RAGChat for: {pdf_path}")
        self.max_history = max_history
        self.chat_history = []  # starts empty

        loader=PyPDFLoader(pdf_path)
        pages=loader.load()
        splitter=RecursiveCharacterTextSplitter(
            chunk_size=500,chunk_overlap=50
        )
        chunks=splitter.split_documents(pages)
        self.embedder=SentenceTransformer("all-MiniLM-L6-v2")
        texts=[c.page_content for c in chunks]
        embeddings=self.embedder.encode(texts)

        #chromadb
        self.client=chromadb.PersistentClient(path="./ragchat_db")
        try:
            self.client.delete_collection("ragchat")
        except:
            pass
        self.collection=self.client.create_collection("ragchat")
        self.collection.add(
            documents=texts,
            embeddings=embeddings.tolist(),
            metadatas=[
                {"page":c.metadata.get("page",0)+1}
                for c in chunks
            ],
            ids=[f"chunk_{i}" for i in range(len(chunks))]
        )

        self.groq=Groq(api_key="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")
        print(f"✅ Ready! {len(chunks)} chunks indexed from {len(pages)} pages")

    def chat(self,question):
        q_emb=self.embedder.encode([question]).tolist()
        results=self.collection.query(
            query_embeddings=q_emb,
            n_results=3,
            include=["documents","metadatas","distances"]
        )
        context=""
        sources=[]
        for i in range(len(results["documents"][0])):
            page=results["metadatas"][0][i]["page"]
            text=results["documents"][0][i]
            relevance=round(1-results["distances"][0][i],2)
            context+=f"[Page{page}]:{text}\n\n"
            sources.append({"page": page, "relevance": relevance})

        system_msg = {
            "role": "system",
            "content": f"""You are a helpful document assistant.
Answer using ONLY the context below. Cite page numbers.
Say "I don't have that information" if context is insufficient.

Context:
{context}"""
        }

        messages = [system_msg] + self.chat_history + [
            {"role": "user", "content": question}
        ]
        
        response = self.groq.chat.completions.create(
            model       = "llama-3.3-70b-versatile",
            messages    = messages,
            max_tokens  = 400,
            temperature = 0.1
        )
        answer = response.choices[0].message.content
        # Update + trim history
        self.chat_history.append(
            {"role": "user",      "content": question}
        )
        self.chat_history.append(
            {"role": "assistant", "content": answer}
        )
        if len(self.chat_history) > self.max_history:
            self.chat_history = self.chat_history[-self.max_history:]

        return {
            "answer":        answer,
            "sources":       sources,
            "history_length": len(self.chat_history)
        }
    def reset(self):
        """Clear conversation history — start fresh"""
        self.chat_history = []
        print("✅ Conversation reset!")

    def get_history(self):
        """Return full conversation history"""
        return self.chat_history


print("\nTesting RAGChat class:")
chat = RAGChat(PDF_PATH)




PART 7: Clean RAGChat Class

Testing RAGChat class:
Initializing RAGChat for: document.pdf


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5233.35it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Ready! 25 chunks indexed from 16 pages


# Notes


In [56]:
# ============================================================
# HOW THE LLM USES HISTORY + CONTEXT
# ============================================================

# The LLM does NOT have permanent memory between API calls.
#
# Every request is stateless:
# meaning the model starts fresh each time.
#
# So to create "memory", we resend:
#
# 1. System prompt
# 2. Retrieved document context
# 3. Previous chat history
# 4. Current user question
#
# together in every API request.

# Example:
#
# messages = [
#     {
#         "role": "system",
#         "content": """
#         You are a helpful assistant.
#
#         Context:
#         [Page 2]: Machine learning is...
#         """
#     },
#
#     {
#         "role": "user",
#         "content": "What is machine learning?"
#     },
#
#     {
#         "role": "assistant",
#         "content": "ML is a subset of AI."
#     },
#
#     {
#         "role": "user",
#         "content": "What are its types?"
#     }
# ]

# Internally, the model reads this like one long conversation:
#
# System:
# You are a helpful assistant.
# Context:
# [Page 2]: Machine learning is...
#
# User:
# What is machine learning?
#
# Assistant:
# ML is a subset of AI.
#
# User:
# What are its types?

# Because the earlier messages are included,
# the model understands:
#
# "its" = machine learning

# The transformer model uses something called
# self-attention.
#
# This allows current words/tokens to look at
# earlier words/tokens in the prompt.
#
# So when the model reads:
#
# "What are its types?"
#
# it attends to earlier conversation and context
# to infer the meaning of "its".

# Retrieved document chunks inside the system prompt
# act as grounding knowledge.
#
# Example:
#
# Context:
# [Page 5]: Supervised learning uses labeled data.
# [Page 6]: Unsupervised learning finds hidden patterns.
#
# The model uses this information while generating answers.

# IMPORTANT:
#
# The LLM does NOT separately process:
# - history
# - context
# - question
#
# It processes ONE combined token sequence.

# Conceptually:
#
# Instructions
# + Retrieved Context
# + Conversation History
# + Current Question
# --------------------------------
# = Final Prompt Sent To LLM

# This is what creates the illusion of memory
# in chatbot applications.
#
# If we only send the latest question:
#
# "What are its types?"
#
# the model cannot know what "its" refers to.
#
# Therefore:
#
# Conversation memory in LLM apps is actually
# re-sending previous conversation context
# inside every request.
# ============================================================

# DAY 3 CHALLENGE


In [57]:
print("\n" + "=" * 55)
print("DAY 3 CHALLENGE")
print("=" * 55)

print("""
1. Have a 10 message conversation with your document.
   Use pronouns like "it", "that", "they", "the first one"
   Does the model always understand the reference correctly?
   When does it get confused?

# Answer:
# The model usually understands references correctly
# when the related topic exists in recent chat history.
#
# Example:
# Q1: What is machine learning?
# Q2: What are its types?
#
# Here "its" = machine learning.
#
# But the model can get confused when:
# - conversation becomes too long
# - multiple topics are discussed
# - history is trimmed
# - pronoun reference is ambiguous
#
# Example:
# If we discussed ML, neural networks, and databases,
# then asking:
# "Which one is faster?"
# may confuse the model because "one" is unclear.


2. Try this: Ask a question, get an answer.
   Then ask "Are you sure about that?"
   Does it correctly refer back to its previous answer?

# Answer:
# Yes, because the previous assistant response is
# included in chat_history and sent again to the LLM.
#
# The model reads:
#
# User: ...
# Assistant: ...
# User: Are you sure about that?
#
# So it understands that "that" refers to its
# immediately previous answer.


3. Change max_history from 10 to 2.
   Have a long conversation.
   At what point does it "forget" earlier context?
   What does this tell you about context window limits?

# Answer:
# The model starts forgetting older information once
# messages are removed from chat_history.
#
# With max_history = 2:
# only the latest 2 messages remain.
#
# So older conversation context disappears.
#
# This teaches an important concept:
#
# LLMs do NOT have permanent memory.
# They only know what is included in the current prompt.
#
# If older messages are not sent again,
# the model cannot remember them.


4. Call chat.reset() mid conversation.
   Ask a follow up question that references something
   said before the reset.
   What happens? Why?

# Answer:
# After chat.reset(), chat_history becomes empty.
#
# So follow-up references fail.
#
# Example:
#
# Before reset:
# Q1: What is ML?
#
# After reset:
# Q2: What are its types?
#
# Now the model may not understand what "its" means.
#
# This happens because the old conversation is no
# longer included in the messages sent to the LLM.


5. Most important — explain in your own words:
   Why do we pass the FULL history to the LLM every time
   instead of just storing it somewhere and only sending
   the latest question?
   This is a key architectural concept in LLM apps.

# Answer:
# LLMs are stateless.
#
# They do not automatically remember previous API calls.
#
# Every request starts fresh.
#
# The ONLY way the model can remember earlier conversation
# is if we send the previous messages again inside the
# current prompt.
#
# So we pass:
#
# - system prompt
# - previous chat history
# - current question
#
# together every time.
#
# This creates the illusion of memory.
#
# If we only send the latest question:
#
# "What are its types?"
#
# the model has no idea what "its" refers to.
#
# Therefore conversation memory in LLM apps is actually:
#
# re-sending previous conversation context in every request.
""")


DAY 3 CHALLENGE

1. Have a 10 message conversation with your document.
   Use pronouns like "it", "that", "they", "the first one"
   Does the model always understand the reference correctly?
   When does it get confused?

# Answer:
# The model usually understands references correctly
# when the related topic exists in recent chat history.
#
# Example:
# Q1: What is machine learning?
# Q2: What are its types?
#
# Here "its" = machine learning.
#
# But the model can get confused when:
# - conversation becomes too long
# - multiple topics are discussed
# - history is trimmed
# - pronoun reference is ambiguous
#
# Example:
# If we discussed ML, neural networks, and databases,
# then asking:
# "Which one is faster?"
# may confuse the model because "one" is unclear.


2. Try this: Ask a question, get an answer.
   Then ask "Are you sure about that?"
   Does it correctly refer back to its previous answer?

# Answer:
# Yes, because the previous assistant response is
# included in chat_history